In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder

In [3]:
big_mart = pd.read_csv('train.csv')
big_mart.columns = big_mart.columns.str.strip()
big_mart.head()

,Item_Identifier,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Identifier,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Outlet_Type,Item_Outlet_Sales
0,FDA15,9.30,Low Fat,0.016047,Dairy,249.8092,OUT049,1999,Medium,Tier 1,Supermarket Type1,3735.1380
1,DRC01,5.92,Regular,0.019278,Soft Drinks,48.2692,OUT018,2009,Medium,Tier 3,Supermarket Type2,443.4228
2,FDN15,17.50,Low Fat,0.016760,Meat,141.6180,OUT049,1999,Medium,Tier 1,Supermarket Type1,2097.2700
3,FDX07,19.20,Regular,0.000000,Fruits and Vegetables,182.0950,OUT010,1998,NaN,Tier 3,Grocery Store,732.3800
4,NCD19,8.93,Low Fat,0.000000,Household,53.8614,OUT013,1987,High,Tier 3,Supermarket Type1,994.7052


In [4]:
print(big_mart.shape)

(8523, 12)


In [5]:
print(big_mart.columns)

Index(['Item_Identifier', 'Item_Weight', 'Item_Fat_Content', 'Item_Visibility',
       'Item_Type', 'Item_MRP', 'Outlet_Identifier',
       'Outlet_Establishment_Year', 'Outlet_Size', 'Outlet_Location_Type',
       'Outlet_Type', 'Item_Outlet_Sales'],
      dtype='str')


In [6]:
big_mart.info()

<class 'pandas.DataFrame'>
RangeIndex: 8523 entries, 0 to 8522
Data columns (total 12 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Item_Identifier            8523 non-null   str    
 1   Item_Weight                7060 non-null   float64
 2   Item_Fat_Content           8523 non-null   str    
 3   Item_Visibility            8523 non-null   float64
 4   Item_Type                  8523 non-null   str    
 5   Item_MRP                   8523 non-null   float64
 6   Outlet_Identifier          8523 non-null   str    
 7   Outlet_Establishment_Year  8523 non-null   int64  
 8   Outlet_Size                6113 non-null   str    
 9   Outlet_Location_Type       8523 non-null   str    
 10  Outlet_Type                8523 non-null   str    
 11  Item_Outlet_Sales          8523 non-null   float64
dtypes: float64(4), int64(1), str(7)
memory usage: 1.2 MB


In [7]:
big_mart.describe()

,Item_Weight,Item_Visibility,Item_MRP,Outlet_Establishment_Year,Item_Outlet_Sales
count,7060.000000,8523.000000,8523.000000,8523.000000,8523.000000
mean,12.857645,0.066132,140.992782,1997.831867,2181.288914
std,4.643456,0.051598,62.275067,8.371760,1706.499616
min,4.555000,0.000000,31.290000,1985.000000,33.290000
25%,8.773750,0.026989,93.826500,1987.000000,834.247400
50%,12.600000,0.053931,143.012800,1999.000000,1794.331000
75%,16.850000,0.094585,185.643700,2004.000000,3101.296400
max,21.350000,0.328391,266.888400,2009.000000,13086.964800


In [8]:
big_mart.isnull().sum()

Item_Identifier                 0
Item_Weight                  1463
Item_Fat_Content                0
Item_Visibility                 0
Item_Type                       0
Item_MRP                        0
Outlet_Identifier               0
Outlet_Establishment_Year       0
Outlet_Size                  2410
Outlet_Location_Type            0
Outlet_Type                     0
Item_Outlet_Sales               0
dtype: int64

In [9]:
big_mart.duplicated().sum()

np.int64(0)

In [10]:
outlet_size_mode = (
    big_mart.pivot_table(
        values = 'Outlet_Size',
        columns = 'Outlet_Type',
        aggfunc = (lambda x: x.mode([0]))
    )
)
print(outlet_size_mode)

Outlet_Type Grocery Store Supermarket Type1 Supermarket Type2 Supermarket Type3
Outlet_Size         Small             Small            Medium            Medium


In [11]:
big_mart['Item_Weight'] = big_mart['Item_Weight'].fillna(big_mart['Item_Weight'].mean())

missing_outlet_size = big_mart['Outlet_Size'].isnull()
big_mart.loc[missing_outlet_size, 'Outlet_Size'] = big_mart.loc[missing_outlet_size, 'Outlet_Type'].apply(
    lambda x : outlet_size_mode[x]
    ['Outlet_Size']
    if x in outlet_size_mode[x]
    else 'Medium'
)

In [12]:
encoder = LabelEncoder()
categorical_columns = [
    'Item_Identifier',
    'Item_Fat_Content',
    'Item_Type',
    'Outlet_Identifier',
    'Outlet_Size',
    'Outlet_Location_Type',
    'Outlet_Type',
]
for col in categorical_columns:
    big_mart[col] = encoder.fit_transform(big_mart[col])
    
big_mart.head()

,Item_Identifier,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Identifier,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Outlet_Type,Item_Outlet_Sales
0,156,9.30,1,0.016047,4,249.8092,9,1999,1,0,1,3735.1380
1,8,5.92,2,0.019278,14,48.2692,3,2009,1,2,2,443.4228
2,662,17.50,1,0.016760,10,141.6180,9,1999,1,0,1,2097.2700
3,1121,19.20,2,0.000000,6,182.0950,0,1998,1,2,0,732.3800
4,1297,8.93,1,0.000000,9,53.8614,1,1987,0,2,1,994.7052


In [13]:
X = big_mart.drop(columns='Item_Outlet_Sales')
y = big_mart['Item_Outlet_Sales']

print('Features shape (X) : ', X.shape)
print('Target shape (y) : ', y.shape)

Features shape (X) :  (8523, 11)
Target shape (y) :  (8523,)


In [14]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size = 0.2, random_state=42
)
print('original data shape : ', X.shape)
print('training data shape : ', X_train.shape)
print('testing data shape : ', X_test.shape)

original data shape :  (8523, 11)
training data shape :  (6818, 11)
testing data shape :  (1705, 11)


In [15]:
from xgboost import XGBRegressor

regressor = XGBRegressor(
    max_depth = 4,
    learning_rate = 0.05,
    n_estimators = 100,
    subsample = 0.8,
    colsample_bytree = 0.8,
    random_state = 42
)

regressor.fit(X_train, y_train)

,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [16]:
from sklearn.metrics import r2_score, mean_absolute_error


In [17]:
train_pred = regressor.predict(X_train)
r2_train = r2_score(y_train, train_pred)
mae_train = mean_absolute_error(y_train, train_pred)
print('r2 score on training data : ', r2_train)
print('mean absolute error for training data : ', mae_train)

r2 score on training data :  0.6241495878521987
mean absolute error for training data :  743.7930170646794


In [18]:
test_pred = regressor.predict(X_test)
r2_test = r2_score(y_test, test_pred)
mae_test = mean_absolute_error(y_test, test_pred)
print('r2 score on test data : ', r2_test)
print('mean absolute error for testing data : ', mae_test)

r2 score on test data :  0.616422427707666
mean absolute error for testing data :  716.6764805522714


In [19]:
test_data = pd.read_csv('test.csv')
test_data.columns = test_data.columns.str.strip()
test_data.head()

,Item_Identifier,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Identifier,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Outlet_Type
0,FDW58,20.750,Low Fat,0.007565,Snack Foods,107.8622,OUT049,1999,Medium,Tier 1,Supermarket Type1
1,FDW14,8.300,reg,0.038428,Dairy,87.3198,OUT017,2007,NaN,Tier 2,Supermarket Type1
2,NCN55,14.600,Low Fat,0.099575,Others,241.7538,OUT010,1998,NaN,Tier 3,Grocery Store
3,FDQ58,7.315,Low Fat,0.015388,Snack Foods,155.0340,OUT017,2007,NaN,Tier 2,Supermarket Type1
4,FDY38,NaN,Regular,0.118599,Dairy,234.2300,OUT027,1985,Medium,Tier 3,Supermarket Type3


In [20]:
test_data.shape

(5681, 11)

In [21]:
test_data.isnull().sum()

Item_Identifier                 0
Item_Weight                   976
Item_Fat_Content                0
Item_Visibility                 0
Item_Type                       0
Item_MRP                        0
Outlet_Identifier               0
Outlet_Establishment_Year       0
Outlet_Size                  1606
Outlet_Location_Type            0
Outlet_Type                     0
dtype: int64

In [22]:
test_data.duplicated().sum()

np.int64(0)

In [23]:
outlet_size_mode_2 = (
    test_data.pivot_table(
        values = 'Outlet_Size',
        columns = 'Outlet_Type',
        aggfunc = (lambda x: x.mode([0]))
    )
)
print(outlet_size_mode_2)

Outlet_Type Grocery Store Supermarket Type1 Supermarket Type2 Supermarket Type3
Outlet_Size         Small             Small            Medium            Medium


In [24]:
test_data['Item_Weight'] = test_data['Item_Weight'].fillna(test_data['Item_Weight'].mean())

missing_outlet_size_2 = test_data['Outlet_Size'].isnull()
test_data.loc[missing_outlet_size_2, 'Outlet_Size'] = test_data.loc[missing_outlet_size_2, 'Outlet_Type'].apply(
    lambda x : outlet_size_mode_2[x]
    ['Outlet_Size']
    if x in outlet_size_mode_2[x]
    else 'Medium'
)

In [25]:
le = LabelEncoder()
categorical_columns = [
    'Item_Identifier',
    'Item_Fat_Content',
    'Item_Type',
    'Outlet_Identifier',
    'Outlet_Size',
    'Outlet_Location_Type',
    'Outlet_Type',
]
for col in categorical_columns:
    test_data[col] = le.fit_transform(test_data[col])
    
test_data.head()

,Item_Identifier,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Identifier,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Outlet_Type
0,1103,20.750000,1,0.007565,13,107.8622,9,1999,1,0,1
1,1067,8.300000,4,0.038428,4,87.3198,2,2007,1,1,1
2,1406,14.600000,1,0.099575,11,241.7538,0,1998,1,2,0
3,809,7.315000,1,0.015388,13,155.0340,2,2007,1,1,1
4,1184,12.695633,2,0.118599,4,234.2300,5,1985,1,2,3


In [26]:
test_data.isnull().sum()

Item_Identifier              0
Item_Weight                  0
Item_Fat_Content             0
Item_Visibility              0
Item_Type                    0
Item_MRP                     0
Outlet_Identifier            0
Outlet_Establishment_Year    0
Outlet_Size                  0
Outlet_Location_Type         0
Outlet_Type                  0
dtype: int64

In [27]:
test_prediction = regressor.predict(test_data)

In [28]:
# 1. Generate the predictions for the test dataset
test_predictions = regressor.predict(test_data)

# 2. Save them into a submission dataframe
submission = pd.DataFrame(
    {
        'Item_Identifier': test_data['Item_Identifier'],
        'Outlet_Identifier': test_data['Outlet_Identifier'],
        'Item_Outlet_Sales': test_predictions,
    }
)

# 3. Export to a CSV file
submission.to_csv('BigMart_Submission.csv', index=False)
print('Test predictions saved successfully!')

Test predictions saved successfully!


In [29]:
from sklearn.model_selection import GridSearchCV
from xgboost import XGBRegressor

model = XGBRegressor(random_state=42)

param_grid = {
    'n_estimators': [100, 300, 500],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [3, 5, 7],
    'subsample': [0.8, 1.0],
}

grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    scoring='r2',  # Using R-squared since this is a regression problem
    cv=3,  # 3-fold cross-validation
    n_jobs=-1,  # Uses all CPU cores to run faster
    verbose=1,
)

# 4. Run the search on your BigMart training data
grid_search.fit(X_train, y_train)

# 5. View the best combination found
print('Best Parameters Found:', grid_search.best_params_)
print('Best Cross-Validation R2 Score:', grid_search.best_score_)

# 6. Automatically use the best model to predict on your test dataset
best_model = grid_search.best_estimator_
test_predictions = best_model.predict(test_data)

Fitting 3 folds for each of 54 candidates, totalling 162 fits


Best Parameters Found: {'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 100, 'subsample': 1.0}
Best Cross-Validation R2 Score: 0.595548385152583


In [30]:
from sklearn.ensemble import RandomForestRegressor

regressor = RandomForestRegressor(n_estimators=100, random_state=42)
regressor.fit(X_train, y_train)
for_pred = regressor.predict(X_test)

In [32]:
r2_for = r2_score(y_test, for_pred)
mae_for = mean_absolute_error(y_test, for_pred)

print(f'r2 score for random forest is {r2_for}')
print(f'mean absolute error for random forest is {mae_for}')

r2 score for random forest is 0.5625997959285007
mean absolute error for random forest is 760.6721186064516


In [41]:
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow.keras import layers, models

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

nn_model = models.Sequential([
    layers.Dense(128, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    layers.Dense(64, activation='relu'),
    layers.Dense(32, activation='relu'),
    layers.Dense(1)
])

c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [42]:
nn_model.compile(
    optimizer='adam',
    loss = 'mean_squared_error',
    metrics = ['mae']
)

In [44]:
history = nn_model.fit(
    X_train_scaled,
    y_train,
    validation_split=0.2,
    epochs = 50,
    batch_size = 32,
    verbose = 1,
)

test_loss, test_mae = nn_model.evaluate(X_test_scaled, y_test)

print(f'test mean absolute error : {test_mae}')
print(f'test mean loss : {test_loss}')

Epoch 1/50
171/171 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 1146973.1250 - mae: 754.8613 - val_loss: 1235318.3750 - val_mae: 776.9099
Epoch 2/50
171/171 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 1145980.6250 - mae: 754.2469 - val_loss: 1232882.6250 - val_mae: 778.3029
Epoch 3/50
171/171 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 1149553.2500 - mae: 754.2540 - val_loss: 1238350.6250 - val_mae: 776.6306
Epoch 4/50
171/171 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 1144855.6250 - mae: 753.2806 - val_loss: 1233112.2500 - val_mae: 777.6175
Epoch 5/50
171/171 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 1145532.3750 - mae: 754.3350 - val_loss: 1245947.2500 - val_mae: 783.1125
Epoch 6/50
171/171 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 1146926.5000 - mae: 753.3344 - val_loss: 1235039.0000 - val_mae: 779.2603
Epoch 7/50
171/171 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 1144101.3750 - mae: 753.5084 - val_loss: 1239843.6250 - val_mae: 776.3226
Epoch 8/50
171/171 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 11